# Homework #2 - InfluxDB

## Caricamento Dati

In [1]:
import random
from influxdb_client import InfluxDBClient, Point, WritePrecision
import influxdb_client
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Setup e connessione al server InfluxDB
org_name = "spamic"
token = "6ANpdrZyP8Nk7HBib5puP7dx8fJrFNG2xCPlBiXqDSKAsSfjKsUMPlgEVvoQMllyP0HCDAlherDBZZA_z42utw=="
url = "http://127.0.0.1:8086"
bucket_name = "lapd_crime_data"

RECREATE_BUCKET_BEFORE_LOAD = False
CHUNK_SIZE = 2000
BATCH_SIZE = 200


MEASUREMENT_NAME = "crime_data"
DAILY_MEASUREMENT_NAME = "crime_daily"

CSV_FILE = "./Crime_Data_from_2020_to_2024.csv"
MAX_ALLOWED_DATE = pd.Timestamp("2025-11-30 23:59:59")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

WRITE_TO_INFLUX = True


USECOLS = [
    "DR_NO", "Date Rptd", "DATE OCC", "TIME OCC", "AREA", "AREA NAME",
    "Rpt Dist No", "Part 1-2", "Crm Cd", "Crm Cd Desc",
    "Vict Age", "Vict Sex", "Vict Descent", "Weapon Used Cd",
    "LAT", "LON"
]

DTYPES = {
    "DR_NO": "string",
    "AREA": "Int16",
    "AREA NAME": "category",
    "Rpt Dist No": "Int32",
    "Part 1-2": "Int8",
    "Crm Cd": "Int32",
    "Crm Cd Desc": "string",
    "Vict Age": "Int16",
    "Vict Sex": "category",
    "Vict Descent": "category",
    "Weapon Used Cd": "string",
    "LAT": "float32",
    "LON": "float32",
    "TIME OCC": "string"
}

VIOLENT_KEYWORDS = [
    "HOMICIDE", "MURDER", "RAPE", "ROBBERY",
    "ASSAULT", "KIDNAPPING", "SHOTS FIRED", "BATTERY"
]


c:\Users\Spazzo\anaconda3\envs\sistemi-evolutivi-big-data\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Manipolazione dati

In [2]:
# Funzioni per estrarre tempo, fascia oraria, stagione, validità dei dati, presenza di armi e violenza
def parse_time_occ(value):
    if pd.isna(value):
        return None, None
    s = str(value).strip()
    if "." in s:
        s = s.split(".")[0]
    s = s.zfill(4)
    try:
        hour = int(s[:2])
        minute = int(s[2:])
        if not (0 <= hour <= 23 and 0 <= minute <= 59):
            return None, None
        return hour, minute
    except Exception:
        return None, None

def get_time_band(hour):
    if hour is None or pd.isna(hour):
        return "unknown"
    return "day" if 6 <= int(hour) < 18 else "night"

def get_season(ts):
    month = ts.month
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"

def is_valid_crime(row):
    if pd.isna(row["Crm Cd"]) or pd.isna(row["Crm Cd Desc"]):
        return False
    if str(row["Crm Cd Desc"]).strip() == "":
        return False
    if row["Part 1-2"] not in [1, 2]:
        return False
    return True

def is_valid_location(row):
    lat = row["LAT"]
    lon = row["LON"]
    if pd.isna(lat) or pd.isna(lon):
        return False
    if float(lat) == 0 or float(lon) == 0:
        return False
    if not (-90 <= float(lat) <= 90):
        return False
    if not (-180 <= float(lon) <= 180):
        return False
    return True

def has_weapon(weapon_code):
    return "yes" if pd.notna(weapon_code) and str(weapon_code).strip() != "" else "no"

def is_violent(desc):
    if pd.isna(desc):
        return 0
    desc = str(desc).upper()
    return int(any(k in desc for k in VIOLENT_KEYWORDS))

def clean_age(age):
    if pd.isna(age):
        return None
    try:
        age = int(age)
        return age if age >= 0 else None
    except Exception:
        return None

def classify(value, warning_threshold, alarm_threshold):
    if value >= alarm_threshold:
        return "ALARM"
    if value >= warning_threshold:
        return "WARNING"
    return "OK"


def combine_alerts(row):
    statuses = {
        row["daily_crimes_status"],
        row["daily_violent_status"],
        row["daily_weapon_status"]
    }
    if "ALARM" in statuses:
        return "ALARM"
    if "WARNING" in statuses:
        return "WARNING"
    return "OK"


def transform_chunk(df):
    df["Date Rptd"] = pd.to_datetime(
        df["Date Rptd"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )
    df["DATE OCC"] = pd.to_datetime(
        df["DATE OCC"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )

    df[["occ_hour", "occ_minute"]] = df["TIME OCC"].apply(
        lambda x: pd.Series(parse_time_occ(x))
    )

    df = df[df["Date Rptd"].notna() & df["DATE OCC"].notna()]
    df = df[(df["Date Rptd"] <= MAX_ALLOWED_DATE) & (df["DATE OCC"] <= MAX_ALLOWED_DATE)]
    df = df[df["occ_hour"].notna()]
    df = df[df.apply(is_valid_crime, axis=1)]
    df = df[df.apply(is_valid_location, axis=1)]

    df["weapon_used"] = df["Weapon Used Cd"].apply(has_weapon)
    df["is_violent"] = df["Crm Cd Desc"].apply(is_violent)
    df["vict_age_clean"] = df["Vict Age"].apply(clean_age)
    df["time_band"] = df["occ_hour"].apply(get_time_band)
    df["season"] = df["DATE OCC"].apply(get_season)
    df["year"] = df["DATE OCC"].dt.year

    # Timestamp giornaliero allineato alle 12:00:00 come richiesto dalla traccia
    df["ts"] = df["DATE OCC"].dt.normalize() + pd.Timedelta(hours=12)
    df["day"] = df["ts"].dt.date

    return df

def build_daily_analytics_chunk(chunk):
    daily = (
        chunk.groupby(
            ["ts", "AREA", "AREA NAME", "season", "time_band", "Part 1-2", "Crm Cd", "year"],
            as_index=False
        )
        .agg(
            crime_count=("DR_NO", "count"),
            violent_count=("is_violent", "sum"),
            weapon_count=("weapon_used", lambda s: (s == "yes").sum()),
            vict_age_sum=("vict_age_clean", lambda s: s.dropna().sum()),
            vict_age_count=("vict_age_clean", lambda s: s.notna().sum())
        )
    )

    daily["weapon_pct"] = daily.apply(
        lambda row: (row["weapon_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
        axis=1
    )
    daily["violent_pct"] = daily.apply(
        lambda row: (row["violent_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
        axis=1
    )

    return daily



def update_daily_state(chunk, daily_state):
    daily_chunk = (
        chunk.groupby("day")
        .agg(
            daily_crimes=("DR_NO", "count"),
            daily_violent_crimes=("is_violent", "sum"),
            daily_weapon_yes=("weapon_used", lambda s: (s == "yes").sum())
        )
        .reset_index()
    )

    for _, row in daily_chunk.iterrows():
        day = row["day"]

        if day not in daily_state:
            daily_state[day] = {
                "daily_crimes": 0,
                "daily_violent_crimes": 0,
                "daily_weapon_yes": 0
            }

        daily_state[day]["daily_crimes"] += int(row["daily_crimes"])
        daily_state[day]["daily_violent_crimes"] += int(row["daily_violent_crimes"])
        daily_state[day]["daily_weapon_yes"] += int(row["daily_weapon_yes"])


def build_daily_metrics_df(daily_state):
    rows = []

    for day, values in daily_state.items():
        total = values["daily_crimes"]
        weapon_yes = values["daily_weapon_yes"]
        weapon_pct = (weapon_yes / total) * 100.0 if total > 0 else 0.0

        rows.append({
            "day": day,
            "daily_crimes": total,
            "daily_violent_crimes": values["daily_violent_crimes"],
            "daily_weapon_pct": weapon_pct
        })

    return pd.DataFrame(rows)


def apply_alerts(daily_df):
    max_daily_crimes = daily_df["daily_crimes"].max()
    max_daily_violent = daily_df["daily_violent_crimes"].max()
    max_daily_weapon_pct = daily_df["daily_weapon_pct"].max()

    crimes_warning = max_daily_crimes * 0.05
    crimes_alarm = max_daily_crimes * 0.10

    violent_warning = max_daily_violent * 0.05
    violent_alarm = max_daily_violent * 0.10

    weapon_warning = max_daily_weapon_pct * 0.05
    weapon_alarm = max_daily_weapon_pct * 0.10

    daily_df["daily_crimes_warning_threshold"] = crimes_warning
    daily_df["daily_crimes_alarm_threshold"] = crimes_alarm
    daily_df["daily_violent_warning_threshold"] = violent_warning
    daily_df["daily_violent_alarm_threshold"] = violent_alarm
    daily_df["daily_weapon_warning_threshold"] = weapon_warning
    daily_df["daily_weapon_alarm_threshold"] = weapon_alarm

    daily_df["daily_crimes_status"] = daily_df["daily_crimes"].apply(
        lambda x: classify(x, crimes_warning, crimes_alarm)
    )
    daily_df["daily_violent_status"] = daily_df["daily_violent_crimes"].apply(
        lambda x: classify(x, violent_warning, violent_alarm)
    )
    daily_df["daily_weapon_status"] = daily_df["daily_weapon_pct"].apply(
        lambda x: classify(x, weapon_warning, weapon_alarm)
    )

    return daily_df


def drop_bucket(client, bucket_name):
    buckets_api = client.buckets_api()
    bucket = buckets_api.find_bucket_by_name(bucket_name)
    print (f"Controllo esistenza bucket '{bucket_name}'...")
    if bucket is not None:
        print(f"Bucket '{bucket_name}' trovato, procedo con l'eliminazione...")
        buckets_api.delete_bucket(bucket)
        print(f"Bucket '{bucket_name}' eliminato.")
    else:
        print(f"Bucket '{bucket_name}' non trovato, niente da eliminare.")


def create_bucket(client, bucket_name, org_name):
    buckets_api = client.buckets_api()
    bucket = buckets_api.find_bucket_by_name(bucket_name)
    if bucket is None:
        buckets_api.create_bucket(bucket_name=bucket_name, org=org_name)
        print(f"Bucket '{bucket_name}' creato.")
    else:
        print(f"Bucket '{bucket_name}' gia esistente.")



In [ ]:
def row_to_point(row):
    p = (
        Point(MEASUREMENT_NAME)
        .tag("dr_no", str(row["DR_NO"]))
        .tag("area_id", str(row["AREA"]))
        .tag("area_name", str(row["AREA NAME"]))
        .tag("rpt_dist_no", str(row["Rpt Dist No"]))
        .tag("part", str(int(row["Part 1-2"])))
        .tag("weapon_used", row["weapon_used"])
        .tag("season", row["season"])
        .tag("time_band", row["time_band"])
        .field("crime_count", 1)
        .field("crime_code", str(row["Crm Cd"]))
        .field("crime_desc", str(row["Crm Cd Desc"]))
        .field("is_violent", int(row["is_violent"]))
        .field("year", int(row["year"]))
        .field("lat", float(row["LAT"]))
        .field("lon", float(row["LON"]))
        .field("occ_hour", int(row["occ_hour"]))
        .field("daily_crimes_current", int(row["daily_crimes"]))
        .field("daily_violent_current", int(row["daily_violent_crimes"]))
        .field("daily_weapon_pct_current", float(row["daily_weapon_pct"]))
        .field("daily_crimes_status", str(row["daily_crimes_status"]))
        .field("daily_violent_status", str(row["daily_violent_status"]))
        .field("daily_weapon_status", str(row["daily_weapon_status"]))
        .field("alert_level", str(row["alert_level"]))
        .time(row["ts"], WritePrecision.NS)
    )

    if pd.notna(row["vict_age_clean"]):
        p = p.field("vict_age", int(row["vict_age_clean"]))

    if pd.notna(row["Weapon Used Cd"]) and str(row["Weapon Used Cd"]).strip() != "":
        try:
            p = p.field("weapon_code", int(float(row["Weapon Used Cd"])))
        except Exception:
            pass

    return p

def daily_row_to_point(row):
    p = (
        Point(DAILY_MEASUREMENT_NAME)
        .tag("area_id", str(row["AREA"]))
        .tag("area_name", str(row["AREA NAME"]))
        .tag("season", row["season"])
        .tag("time_band", row["time_band"])
        .tag("part", str(int(row["Part 1-2"])))
        .tag("crime_code", str(row["Crm Cd"]))
        .tag("year", str(int(row["year"])))
        .field("crime_count", int(row["crime_count"]))
        .field("violent_count", int(row["violent_count"]))
        .field("weapon_count", int(row["weapon_count"]))
        .field("weapon_pct", float(row["weapon_pct"]))
        .field("violent_pct", float(row["violent_pct"]))
        .field("vict_age_sum", float(row["vict_age_sum"]))
        .field("vict_age_count", int(row["vict_age_count"]))
        .time(row["ts"], WritePrecision.NS)
    )
    return p



def flush_points(write_api, bucket_name, rows):
    batch = []
    for _, row in rows.iterrows():
        batch.append(row_to_point(row))
        if len(batch) >= BATCH_SIZE:
            write_api.write(bucket=bucket_name, record=batch)
            batch.clear()

    if batch:
        write_api.write(bucket=bucket_name, record=batch)


def flush_daily_points(write_api, bucket_name, rows):
    batch = []
    for _, row in rows.iterrows():
        batch.append(daily_row_to_point(row))
        if len(batch) >= BATCH_SIZE:
            write_api.write(bucket=bucket_name, record=batch)
            batch.clear()

    if batch:
        write_api.write(bucket=bucket_name, record=batch)



cleaned_csv = OUTPUT_DIR / "cleaned_data.csv"
alerts_csv = OUTPUT_DIR / "daily_alerts.csv"

first_write_cleaned = True
daily_state = {}
total_raw = 0
total_valid = 0
daily_analytics_parts = []

total_rows = sum(1 for _ in open(CSV_FILE, "r", encoding="utf-8")) - 1

if WRITE_TO_INFLUX:
    client = InfluxDBClient(url=url, token=token, org=org_name, timeout=240000)

    if RECREATE_BUCKET_BEFORE_LOAD:
        drop_bucket(client, bucket_name)
        create_bucket(client, bucket_name, org_name)

    write_api = client.write_api()
else:
    client = None
    write_api = None

reader = pd.read_csv(CSV_FILE, usecols=USECOLS, dtype=DTYPES, chunksize=CHUNK_SIZE, encoding="utf-8")

with tqdm(total=total_rows, desc="Caricamento crimini", unit="righe") as pbar:
    for chunk_idx, chunk in enumerate(reader, start=1):
        raw_chunk_len = len(chunk)
        total_raw += raw_chunk_len

        chunk = transform_chunk(chunk)

        if len(chunk) == 0:
            pbar.update(raw_chunk_len)
            print(f"Chunk {chunk_idx}: nessun record valido")
            continue

        total_valid += len(chunk)

        update_daily_state(chunk, daily_state)
        daily_df = build_daily_metrics_df(daily_state)
        daily_df = apply_alerts(daily_df)

        chunk = chunk.merge(daily_df, on="day", how="left")
        chunk["alert_level"] = chunk.apply(combine_alerts, axis=1)
        daily_analytics_chunk = build_daily_analytics_chunk(chunk)
        daily_analytics_parts.append(daily_analytics_chunk)


        chunk.to_csv(
            cleaned_csv,
            mode="w" if first_write_cleaned else "a",
            header=first_write_cleaned,
            index=False
        )
        first_write_cleaned = False

        if WRITE_TO_INFLUX:
            flush_points(write_api, bucket_name, chunk)



        pbar.update(raw_chunk_len)
        pbar.set_postfix({
            "chunk": chunk_idx,
            "validi": len(chunk)
        })

        print(
            f"Chunk {chunk_idx}: raw={raw_chunk_len} validi_scritti={len(chunk)} "
            f"mem={chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
        )


if daily_analytics_parts:
    final_daily_analytics = pd.concat(daily_analytics_parts, ignore_index=True)

    final_daily_analytics = (
        final_daily_analytics.groupby(
            ["ts", "AREA", "AREA NAME", "season", "time_band", "Part 1-2", "Crm Cd", "year"],
            as_index=False
        )
        .agg(
            crime_count=("crime_count", "sum"),
            violent_count=("violent_count", "sum"),
            weapon_count=("weapon_count", "sum"),
            vict_age_sum=("vict_age_sum", "sum"),
            vict_age_count=("vict_age_count", "sum")
        )
    )

    final_daily_analytics["weapon_pct"] = final_daily_analytics.apply(
        lambda row: (row["weapon_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
        axis=1
    )
    final_daily_analytics["violent_pct"] = final_daily_analytics.apply(
        lambda row: (row["violent_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
        axis=1
    )

    print(f"Aggregati giornalieri finali da scrivere: {len(final_daily_analytics)}")

    if WRITE_TO_INFLUX:
        flush_daily_points(write_api, bucket_name, final_daily_analytics)
    
    print("Scrittura crime_daily completata")

   

if client is not None:
    client.close()

final_daily = build_daily_metrics_df(daily_state)
final_daily = apply_alerts(final_daily)
final_daily.to_csv(alerts_csv, index=False)

print("Caricamento completato")
print("Record grezzi letti:", total_raw)
print("Record validi scritti:", total_valid)
print("CSV pulito salvato in:", cleaned_csv)
print("CSV alert giornalieri salvato in:", alerts_csv)

final_daily.head()


In [3]:
import pandas as pd
from influxdb_client import InfluxDBClient, Point, WritePrecision


DAILY_MEASUREMENT_NAME = "crime_daily"
CLEANED_CSV = "./output/cleaned_data.csv"
BATCH_SIZE = 1000

df = pd.read_csv(CLEANED_CSV)

df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["vict_age_clean"] = pd.to_numeric(df["vict_age_clean"], errors="coerce")
df["is_violent"] = pd.to_numeric(df["is_violent"], errors="coerce").fillna(0).astype(int)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["Crm Cd"] = pd.to_numeric(df["Crm Cd"], errors="coerce").astype("Int64")
df["Part 1-2"] = pd.to_numeric(df["Part 1-2"], errors="coerce").astype("Int64")

daily = (
    df.groupby(
        ["ts", "AREA", "AREA NAME", "season", "time_band", "Part 1-2", "Crm Cd", "year"],
        as_index=False
    )
    .agg(
        crime_count=("DR_NO", "count"),
        violent_count=("is_violent", "sum"),
        weapon_count=("weapon_used", lambda s: (s == "yes").sum()),
        vict_age_sum=("vict_age_clean", lambda s: s.dropna().sum()),
        vict_age_count=("vict_age_clean", lambda s: s.notna().sum())
    )
)

daily["weapon_pct"] = daily.apply(
    lambda row: (row["weapon_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
    axis=1
)
daily["violent_pct"] = daily.apply(
    lambda row: (row["violent_count"] / row["crime_count"]) * 100.0 if row["crime_count"] > 0 else 0.0,
    axis=1
)

def daily_row_to_point(row):
    return (
        Point(DAILY_MEASUREMENT_NAME)
        .tag("area_id", str(row["AREA"]))
        .tag("area_name", str(row["AREA NAME"]))
        .tag("season", str(row["season"]))
        .tag("time_band", str(row["time_band"]))
        .tag("part", str(int(row["Part 1-2"])))
        .tag("crime_code", str(int(row["Crm Cd"])))
        .tag("year", str(int(row["year"])))
        .field("crime_count", int(row["crime_count"]))
        .field("violent_count", int(row["violent_count"]))
        .field("weapon_count", int(row["weapon_count"]))
        .field("weapon_pct", float(row["weapon_pct"]))
        .field("violent_pct", float(row["violent_pct"]))
        .field("vict_age_sum", float(row["vict_age_sum"]))
        .field("vict_age_count", int(row["vict_age_count"]))
        .time(row["ts"], WritePrecision.NS)
    )

client = InfluxDBClient(url=url, token=token, org=org_name, timeout=240000)
write_api = client.write_api()

batch = []
for _, row in daily.iterrows():
    batch.append(daily_row_to_point(row))
    if len(batch) >= BATCH_SIZE:
        write_api.write(bucket=bucket_name, record=batch)
        batch.clear()

if batch:
    write_api.write(bucket=bucket_name, record=batch)

client.close()

print(f"Scritti {len(daily)} punti nella measurement '{DAILY_MEASUREMENT_NAME}'")


Scritti 677067 punti nella measurement 'crime_daily'


In [4]:
from influxdb_client import InfluxDBClient

client = InfluxDBClient(url=url, token=token, org=org_name)
query_api = client.query_api()

q = f'''
import "influxdata/influxdb/schema"
schema.measurements(bucket: "{bucket_name}")
'''

print(query_api.query_data_frame(q))
client.close()


c:\Users\Spazzo\anaconda3\envs\sistemi-evolutivi-big-data\Lib\site-packages\influxdb_client\client\warnings.py:31: MissingPivotFunction: The query doesn't contains the pivot() function.

The result will not be shaped to optimal processing by pandas.DataFrame. Use the pivot() function by:

    
import "influxdata/influxdb/schema"
schema.measurements(bucket: "lapd_crime_data")
 |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")

You can disable this warning by:
    import warnings
    from influxdb_client.client.warnings import MissingPivotFunction

    warnings.simplefilter("ignore", MissingPivotFunction)

For more info see:
    - https://docs.influxdata.com/resources/videos/pivots-in-flux/
    - https://docs.influxdata.com/flux/latest/stdlib/universe/pivot/
    - https://docs.influxdata.com/flux/latest/stdlib/influxdata/influxdb/schema/fieldsascols/

  warnings.warn(message, MissingPivotFunction)


    result  table       _value
0  _result      0  crime_daily
1  _result      0   crime_data
